# Finance Club, IIT Roorkee: Open Projects 2026
## Project: Stochastic Interest Rate Modelling and Prediction
### Implementing, Calibrating, and Extending the Cox-Ingersoll-Ross Model on Real Yield Curve Data

**Author**: Quantitative Research Analyst  
**Date**: May 2026  
**Deliverable**: Single Self-Contained Google Colab Notebook

---

### Project Overview
Interest rates are the fundamental building blocks of the global financial system, dictating the valuation of trillions of dollars in derivatives, bond portfolios, and risk-management strategies. This project implements, calibrates, and extends the famous **Cox-Ingersoll-Ross (CIR) model** using real-world daily yield curve data. 

Our objectives are:
1. **Data Engineering & Preprocessing**: Clean a noisy interest rate dataset, handle formatting anomalies (like leading spaces in column headers), interpolate missing data, and apply robust outlier filtering using a rolling Median Absolute Deviation (MAD) algorithm.
2. **Base CIR Calibration**: Implement and compare **Time-Series Ordinary Least Squares (OLS)**, **Time-Series Maximum Likelihood Estimation (MLE)**, and **Cross-Sectional Yield Curve Calibration (CS)** to find the parameters $(\kappa, 	heta, \sigma)$.
3. **Out-of-Sample Prediction**: Reconstruct the entire yield curve (6M through 2Y) out-of-sample using *only* the 3-Month (3M) rate on each test day, aiming for a pooled $R^2 > 0.85$.
4. **Model Extension**: Research, implement, and backtest the **CIR++ shifted model**, analyzing its performance under structural interest rate regime shifts.
5. **Critical Analysis**: Provide deep theoretical and practical insights addressing the key project questions.


## 1. Core Mathematical Framework

### 1.1 The Cox-Ingersoll-Ross (CIR) Model SDE
The instantaneous short rate $r_t$ under the historical probability measure $\mathbb{P}$ is described by the stochastic differential equation:
$$dr_t = \kappa(	heta - r_t) dt + \sigma \sqrt{r_t} dW_t$$
where:
- $\kappa > 0$ is the **speed of mean reversion**, governing how quickly the short rate is pulled back to its long-run average.
- $	heta > 0$ is the **long-run mean** level of the short rate.
- $\sigma > 0$ is the **volatility coefficient**, representing the amplitude of interest rate fluctuations.
- $W_t$ is a standard Brownian motion under $\mathbb{P}$.

The square-root diffusion term $\sigma \sqrt{r_t}$ ensures that interest rates remain strictly positive. If the rate touches zero, the drift term pulls it back up, provided the **Feller Condition** is satisfied:
$$2 \kappa 	heta \ge \sigma^2$$
If the Feller condition holds, the boundary at $0$ is inaccessible. If it is violated, the boundary is reflective (the rate can touch $0$ but will immediately bounce back).

### 1.2 Zero-Coupon Bond Pricing under CIR
Under a risk-neutral probability measure $\mathbb{Q}$ (accounting for the market price of risk $\lambda$), the price at time $t$ of a zero-coupon bond maturing at $T$ is given by:
$$P(t, T) = A(t, T) e^{-B(t, T) r_t}$$
where $	au = T - t$ is the time-to-maturity, and $A(t, T)$ and $B(t, T)$ are deterministic functions of the model parameters:
$$A(t, T) = \left[ \frac{2 h e^{(\kappa^* + h)\tau/2}}{2 h + (\kappa^* + h)(e^{h\tau} - 1)} \right]^{\frac{2\kappa\theta}{\sigma^2}}$$
$$B(t, T) = \frac{2(e^{h\tau} - 1)}{2 h + (\kappa^* + h)(e^{h\tau} - 1)}$$
$$h = \sqrt{(\kappa^*)^2 + 2\sigma^2}$$
Here, $\kappa^* = \kappa + \lambda$ represents the **risk-neutral speed of mean reversion**. In this notebook, we calibrate these pricing parameters directly to the yield curve.

The continuously compounded yield for maturity $	au$ is derived by taking the negative logarithm of the bond price divided by the maturity $	au$:
$$y(t, \tau) = -\frac{\ln P(t, T)}{\tau} = \frac{B(t, T) r_t - \ln A(t, T)}{\tau}$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import os
import warnings
warnings.filterwarnings('ignore')

# Para cumplir con las reglas del proyecto, todos los comentarios del código deben explicar el porqué en español.
# Configuramos el tema de seaborn para generar gráficos elegantes y con estética premium en los reportes.
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print('Libraries successfully imported!')


In [ ]:
def cir_bond_price_formula(rt, tau, kappa, theta, sigma):
    # Evaluamos analíticamente el precio de un bono cupón cero bajo el modelo CIR.
    # Evitamos divisiones por cero o valores indeterminados limitando los valores inferiores de kappa y sigma.
    sigma = max(sigma, 1e-6)
    kappa = max(kappa, 1e-6)
    
    # Calculamos la variable auxiliar h para incorporar la varianza de la tasa en la estructura temporal.
    h = np.sqrt(kappa**2 + 2 * sigma**2)
    exp_h_tau = np.exp(h * tau)
    
    # Resolvemos el denominador común para A y B, asegurando estabilidad numérica ante vencimientos largos.
    den = 2 * h + (kappa + h) * (exp_h_tau - 1)
    num_A = 2 * h * np.exp((kappa + h) * tau / 2)
    
    # Elevamos a la potencia requerida. Tomamos valor absoluto del cociente para mitigar errores de redondeo en flotantes.
    A = (num_A / den) ** (2 * kappa * theta / sigma**2)
    B = 2 * (exp_h_tau - 1) / den
    
    # Retornamos el precio teórico y los factores de descuento.
    price = A * np.exp(-B * rt)
    return price, A, B

def cir_yield_formula(rt, tau, kappa, theta, sigma):
    # Derivamos el rendimiento teórico de forma continua a partir de la fórmula del precio del bono.
    price, A, B = cir_bond_price_formula(rt, tau, kappa, theta, sigma)
    # Aplicamos un límite inferior (floor) al precio para evitar valores no numéricos (NaN) al calcular el logaritmo.
    price = np.clip(price, 1e-10, None)
    return -np.log(price) / tau

print('Core mathematical pricing functions defined!')


## 2. Milestone A: Data Engineering & Preprocessing

Real-world daily yield curves are notorious for data quality issues, such as missing values, decimal format inconsistencies, outliers from database errors, and leading spaces in column headers. This section implements a robust preprocessing pipeline to clean the datasets:
1. **Column Stripping**: Removes leading/trailing whitespaces in column names (e.g., resolving `' ZC025YR'` to `'ZC025YR'`).
2. **Numeric Coercion & Interpolation**: Formats all columns as numeric, turning text anomalies into `NaN`s, and interpolating them using linear daily interpolation.
3. **Outlier Filtering**: Outliers are identified using a **rolling Median Absolute Deviation (MAD)** window. Rolling MAD is highly robust to clusters of outliers. Any daily rate that deviates from its window's median by more than $5 \times \text{rolling MAD}$ is replaced by the median.


In [ ]:
# Definimos las rutas de los conjuntos de datos en el espacio de trabajo local.
train_path = 'train_data.csv'
test_path = 'test_data.csv'
test_3m_path = 'test_data_3M.csv'

# Cargamos los datos crudos a dataframes de pandas.
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
test_3m_df = pd.read_csv(test_3m_path)

# Eliminamos espacios en blanco accidentales en los encabezados para solucionar la inconsistencia de nombres como ' ZC025YR'.
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()
test_3m_df.columns = test_3m_df.columns.str.strip()

# Convertimos las columnas de fecha a tipo datetime para poder realizar análisis temporales ordenados.
train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])
test_3m_df['Date'] = pd.to_datetime(test_3m_df['Date'])

# Ordenamos cronológicamente los datos para evitar distorsiones en las diferencias de series temporales.
train_df = train_df.sort_values('Date').reset_index(drop=True)
test_df = test_df.sort_values('Date').reset_index(drop=True)
test_3m_df = test_3m_df.sort_values('Date').reset_index(drop=True)

print(f'Training shape: {train_df.shape}, Test shape: {test_df.shape}')

# Función robusta de limpieza y filtrado de valores atípicos.
def clean_data(df, columns):
    df_clean = df.copy()
    for col in columns:
        # Forzamos la conversión a numérico para eliminar cadenas de texto o formatos corruptos.
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        # Rellenamos cualquier valor faltante mediante interpolación lineal diaria para mantener continuidad matemática.
        df_clean[col] = df_clean[col].interpolate(method='linear')
        
        # Configuramos un filtro de desviación mediana absoluta (MAD) móvil para identificar picos de datos erróneos.
        window = 15
        rolling_median = df_clean[col].rolling(window=window, min_periods=1, center=True).median()
        rolling_mad = (df_clean[col] - rolling_median).abs().rolling(window=window, min_periods=1, center=True).median()
        
        # Aplicamos una tolerancia de 5 veces el MAD, con un piso de 10 puntos básicos para series excesivamente estables.
        threshold = np.clip(5 * rolling_mad, 0.001, None)
        
        # Reemplazamos los valores que superen el umbral con la mediana móvil para suavizar anomalías sin distorsionar la tendencia.
        outliers = (df_clean[col] - rolling_median).abs() > threshold
        if outliers.sum() > 0:
            print(f'  [Cleaning] Replaced {outliers.sum()} outliers in {col} with rolling median.')
            df_clean.loc[outliers, col] = rolling_median[outliers]
            
    return df_clean

train_cols = [c for c in train_df.columns if c != 'Date']
test_cols = [c for c in test_df.columns if c != 'Date']

print('Preprocessing Training Data...')
train_clean = clean_data(train_df, train_cols)
print('Preprocessing Test Data...')
test_clean = clean_data(test_df, test_cols)


In [ ]:
# Graficamos la comparación entre los datos crudos y limpios para demostrar visualmente el impacto del filtro MAD.
col_to_plot = 'ZC100YR'
plt.figure(figsize=(12, 5))
plt.plot(train_df['Date'], train_df[col_to_plot], label='Raw Data (with outliers)', color='red', alpha=0.5)
plt.plot(train_clean['Date'], train_clean[col_to_plot], label='Cleaned Data', color='teal', linewidth=1.5)
plt.title(f'Milestone A: Outlier Filter and Preprocessing Demonstration ({col_to_plot})')
plt.xlabel('Date')
plt.ylabel('Yield')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Milestone B: Base CIR Model Calibration

We implement and contrast three different parameter estimation methods. We use the **3-Month yield (ZC025YR)** as our proxy for the instantaneous short rate $r_t$.

### 3.1 Ordinary Least Squares (OLS) on Discretized SDE
By discretizing the SDE using Euler discretization over $\Delta t = 1/252$:
$$r_{t+1} - r_t = \kappa(	heta - r_t)\Delta t + \sigma \sqrt{r_t} \epsilon_{t+1} \sqrt{\Delta t}$$
Dividing both sides by $\sqrt{r_t}$ normalizes the variance:
$$\frac{r_{t+1} - r_t}{\sqrt{r_t}} = \frac{\kappa\theta\Delta t}{\sqrt{r_t}} - \kappa\sqrt{r_t}\Delta t + \sigma\sqrt{\Delta t}\epsilon_{t+1}$$
This is a linear regression without an intercept of the form $Y = \beta_1 X_1 + \beta_2 X_2$, where $Y = \frac{r_{t+1} - r_t}{\sqrt{r_t}}$, $X_1 = \frac{\Delta t}{\sqrt{r_t}}$, and $X_2 = -\sqrt{r_t}\Delta t$.
From the estimated coefficients, we obtain:
$$\kappa = \beta_2, \quad \theta = \frac{\beta_1}{\kappa}, \quad \sigma = \sqrt{\frac{\text{Var}(\text{residuals})}{\Delta t}}$$

### 3.2 Maximum Likelihood Estimation (MLE) using Euler-Maruyama Approximation
Since daily interest rate data is highly frequent, the conditional transition density $r_t \mid r_{t-1}$ can be extremely well-approximated by a Normal distribution (avoiding the massive speed and convergence issues of the Scipy infinite Bessel series for `ncx2.pdf`):
$$r_t \mid r_{t-1} \sim N\left( r_{t-1} + \kappa(\theta - r_{t-1})\Delta t, \sigma^2 r_{t-1} \Delta t \right)$$
We minimize the negative log-likelihood function:
$$\min_{\kappa, \theta, \sigma} \frac{1}{2} \sum_{t=1}^{N-1} \left( \ln(2\pi \sigma^2 r_t \Delta t) + \frac{(r_{t+1} - r_t - \kappa(\theta - r_t)\Delta t)^2}{\sigma^2 r_t \Delta t} \right)$$

### 3.3 Cross-Sectional Yield Curve Calibration (CS)
Rather than looking only at the time series of the short rate, we calibrate the model to fit the entire yield curve at once. This estimates the **risk-neutral** parameters under the $\mathbb{Q}$-measure, directly minimizing the squared differences between the actual yields and the model-implied yields:
$$\min_{\kappa, \theta, \sigma} \sum_{t=1}^N \sum_{j=1}^M \left( y^{model}(t, \tau_j; \kappa, \theta, \sigma) - y^{act}(t, \tau_j) \right)^2$$


In [ ]:
# Mapeamos los plazos en años correspondientes a cada columna de la curva de rendimientos de entrenamiento.
train_maturities = {
    'ZC025YR': 0.25, 'ZC050YR': 0.50, 'ZC075YR': 0.75, 'ZC100YR': 1.00,
    'ZC200YR': 2.00, 'ZC500YR': 5.00, 'ZC1000YR': 10.00, 'ZC2000YR': 20.00,
    'ZC3000YR': 30.00
}

# 1. Método de Mínimos Cuadrados Ordinarios (OLS)
def calibrate_ts_ols(r, dt=1/252.0):
    # Transformamos algebraicamente la SDE discretizada para formular una regresión lineal sin intercepto.
    dr = np.diff(r)
    r_prev = r[:-1]
    
    # Normalizamos por la raíz de r_t para homogeneizar la varianza del error.
    y = dr / np.sqrt(r_prev)
    x1 = dt / np.sqrt(r_prev)
    x2 = -np.sqrt(r_prev) * dt
    
    # Resolvemos el sistema lineal por mínimos cuadrados mínimos.
    X = np.column_stack((x1, x2))
    beta, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)
    beta1, beta2 = beta
    
    # Obtenemos los parámetros históricos.
    kappa = beta2
    theta = beta1 / kappa if kappa != 0 else 0
    
    # Derivamos la volatilidad diaria de los residuos normalizados.
    resids = y - (beta1 * x1 + beta2 * x2)
    sigma = np.sqrt(np.var(resids) / dt)
    
    return kappa, theta, sigma

# 2. Método de Máxima Verosimilitud (MLE) mediante aproximación Normal
def cir_neg_log_likelihood(params, r, dt=1/252.0):
    # Evaluamos la densidad de transición conjunta aproximándola por distribuciones normales locales rápidas.
    kappa, theta, sigma = params
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10
    
    r_prev = r[:-1]
    r_curr = r[1:]
    
    # Definimos la media y varianza condicional de la aproximación de Euler-Maruyama.
    mean = r_prev + kappa * (theta - r_prev) * dt
    var = (sigma**2) * r_prev * dt
    
    # Limitamos la varianza inferiormente para evitar inestabilidades de división y logaritmo.
    var = np.clip(var, 1e-9, None)
    
    # Retornamos la función de coste de máxima verosimilitud negativa.
    nll = 0.5 * np.sum(np.log(2 * np.pi * var) + ((r_curr - mean)**2) / var)
    return nll

# 3. Método de Calibración Transversal (Cross-Sectional)
def cross_sectional_error(params, df_clean, maturities_dict):
    # Optimizamos los parámetros para que el modelo minimice el error de precios en toda la curva transversal.
    kappa, theta, sigma = params
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10
    
    total_error = 0.0
    r_vals = df_clean['ZC025YR'].values  # Tomamos la tasa a 3 meses como proxy de la tasa corta instantánea.
    
    # Calculamos el error cuadrático medio acumulado de todos los plazos.
    for col, tau in maturities_dict.items():
        actual_yields = df_clean[col].values
        pred_yields = cir_yield_formula(r_vals, tau, kappa, theta, sigma)
        total_error += np.sum((actual_yields - pred_yields)**2)
        
    return total_error


In [ ]:
r_train = train_clean['ZC025YR'].values
dt = 1/252.0

# Ejecutamos la calibración por OLS sobre la serie temporal.
kappa_ols, theta_ols, sigma_ols = calibrate_ts_ols(r_train, dt)

# Ejecutamos la calibración por MLE imponiendo cotas físicas de positividad en los parámetros.
init_guess = [max(kappa_ols, 0.01), max(theta_ols, 0.01), max(sigma_ols, 0.01)]
bounds = ((1e-3, 10.0), (1e-3, 0.25), (1e-3, 0.5))
res_mle = minimize(cir_neg_log_likelihood, init_guess, args=(r_train, dt), bounds=bounds, method='L-BFGS-B')
kappa_mle, theta_mle, sigma_mle = res_mle.x if res_mle.success else (kappa_ols, theta_ols, sigma_ols)

# Ejecutamos la calibración transversal (CS) excluyendo la tasa de 3 meses para evitar acoplamiento perfecto.
train_dep_maturities = {k: v for k, v in train_maturities.items() if k != 'ZC025YR'}
res_cs = minimize(cross_sectional_error, init_guess, args=(train_clean, train_dep_maturities), bounds=bounds, method='L-BFGS-B')
kappa_cs, theta_cs, sigma_cs = res_cs.x if res_cs.success else (kappa_ols, theta_ols, sigma_ols)

# Validamos e imprimimos las condiciones de Feller de cada modelo calibrado.
print('='*60)
print('CIR BASE MODEL CALIBRATION SUMMARY')
print('='*60)
print(f'TS OLS:  kappa = {kappa_ols:.6f}, theta = {theta_ols:.6f}, sigma = {sigma_ols:.6f}')
print(f'   Feller satisfied OLS: {2*kappa_ols*theta_ols >= sigma_ols**2} (2*kappa*theta = {2*kappa_ols*theta_ols:.6f}, sigma^2 = {sigma_ols**2:.6f})')
print(f'TS MLE:  kappa = {kappa_mle:.6f}, theta = {theta_mle:.6f}, sigma = {sigma_mle:.6f}')
print(f'   Feller satisfied MLE: {2*kappa_mle*theta_mle >= sigma_mle**2} (2*kappa*theta = {2*kappa_mle*theta_mle:.6f}, sigma^2 = {sigma_mle**2:.6f})')
print(f'CS:      kappa = {kappa_cs:.6f}, theta = {theta_cs:.6f}, sigma = {sigma_cs:.6f}')
print(f'   Feller satisfied CS:  {2*kappa_cs*theta_cs >= sigma_cs**2} (2*kappa*theta = {2*kappa_cs*theta_cs:.6f}, sigma^2 = {sigma_cs**2:.6f})')
print('='*60)


## 4. Milestone C: Out-of-Sample Prediction & Yield Curve Construction

This is the core test of our model's predictive power. For any given day in the out-of-sample test period, our prediction algorithm is **only permitted to ingest the 3-Month (3M) yield** for that day as a proxy for the instantaneous short rate $r_t$. 

Using our calibrated parameters and this 3M rate, we theoretically reconstruct the entire yield curve (6M, 9M, 1Y, and 2Y) and compare it against the held-out actuals in the test set. We evaluate the out-of-sample accuracy using **$R^2$**, **MAE**, and **RMSE**.


In [ ]:
# Mapeamos los vencimientos correspondientes al conjunto de prueba para el cálculo de bond pricing.
test_maturities = {
    'ZC025YR': 0.25, 'ZC050YR': 0.50, 'ZC075YR': 0.75, 'ZC100YR': 1.00,
    'ZC200YR': 2.00
}

def evaluate_model(df_test, maturities_dict, kappa, theta, sigma, label):
    # Evaluamos las predicciones fuera de muestra calculando el R2 agrupado y las métricas de error.
    r_test = df_test['ZC025YR'].values
    total_ss_res = 0.0
    total_ss_tot = 0.0
    
    results = []
    for col, tau in maturities_dict.items():
        if col == 'ZC025YR':
            continue
            
        actual = df_test[col].values
        pred = cir_yield_formula(r_test, tau, kappa, theta, sigma)
        
        # Calculamos la suma de residuos cuadráticos y la suma total de diferencias.
        ss_res = np.sum((actual - pred)**2)
        ss_tot = np.sum((actual - np.mean(actual))**2)
        
        r2 = 1 - (ss_res / ss_tot)
        mae = np.mean(np.abs(actual - pred))
        rmse = np.sqrt(np.mean((actual - pred)**2))
        
        results.append({
            'Tenor': col,
            'Maturity': tau,
            'R2': r2,
            'MAE': mae,
            'RMSE': rmse
        })
        
        total_ss_res += ss_res
        total_ss_tot += ss_tot
        
    # Derivamos el R2 agrupado final.
    pooled_r2 = 1 - (total_ss_res / total_ss_tot)
    return pooled_r2, pd.DataFrame(results)

r2_ols, df_ols = evaluate_model(test_clean, test_maturities, kappa_ols, theta_ols, sigma_ols, 'TS OLS')
r2_mle, df_mle = evaluate_model(test_clean, test_maturities, kappa_mle, theta_mle, sigma_mle, 'TS MLE')
r2_cs, df_cs = evaluate_model(test_clean, test_maturities, kappa_cs, theta_cs, sigma_cs, 'CS')

print('='*60)
print('POOLED OUT-OF-SAMPLE R2 COMPARISON')
print('='*60)
print(f'1. TS OLS Base Pooled R2: {r2_ols:.6f}')
print(f'2. TS MLE Base Pooled R2: {r2_mle:.6f}')
print(f'3. CS Base Pooled R2:     {r2_cs:.6f}')
print('='*60)
print('\nDetailed breakdown of CS Base performance by Tenor:')
print(df_cs.to_string(index=False))


In [ ]:
# Graficamos la comparación temporal para evaluar el ajuste dinámico fuera de muestra en el plazo de 1 año.
r_test = test_clean['ZC025YR'].values
pred_1y = cir_yield_formula(r_test, 1.0, kappa_cs, theta_cs, sigma_cs)

plt.figure(figsize=(12, 6))
plt.plot(test_clean['Date'], test_clean['ZC100YR'], label='Actual 1Y Yield (Held-out)', color='black', linewidth=1.5)
plt.plot(test_clean['Date'], pred_1y, label='CIR Predicted 1Y Yield (CS Calibrated)', color='dodgerblue', linestyle='--')
plt.title('Milestone C: Out-of-Sample 1Y Yield Prediction Over Time')
plt.xlabel('Date')
plt.ylabel('Yield')
plt.legend()
plt.tight_layout()
plt.show()

# Seleccionamos una fecha del test set para graficar una instantánea transversal de la curva reconstruida.
test_date_idx = 100
date_str = test_clean['Date'].iloc[test_date_idx].strftime('%Y-%m-%d')
rt_val = test_clean['ZC025YR'].iloc[test_date_idx]

actual_curve = [test_clean[col].iloc[test_date_idx] for col in ['ZC025YR', 'ZC050YR', 'ZC075YR', 'ZC100YR', 'ZC200YR']]
pred_curve = [rt_val] + [cir_yield_formula(rt_val, tau, kappa_cs, theta_cs, sigma_cs) for tau in [0.50, 0.75, 1.00, 2.00]]
maturities_curve = [0.25, 0.50, 0.75, 1.00, 2.00]

plt.figure(figsize=(8, 5))
plt.plot(maturities_curve, actual_curve, 'ko-', label=f'Actual Curve ({date_str})')
plt.plot(maturities_curve, pred_curve, 'bs--', label='Predicted Curve (CIR CS)')
plt.title(f'Yield Curve Reconstruction on {date_str}')
plt.xlabel('Maturity (Years)')
plt.ylabel('Yield')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Milestone D: Model Extension (CIR++)

The standard single-factor CIR model has well-documented limitations: it constraints the yield curve to shapes that a single factor can produce and cannot perfectly fit the initial yield curve. We implement the **CIR++ (Shifted CIR)** extension, which introduces a deterministic shift parameter $\phi(\tau)$ for each maturity tenor $\tau$:
$$r_t = x_t + \phi(t)$$
The out-of-sample predicted yield for maturity $\tau$ is given by:
$$y^{pred}(t, \tau) = y_{CIR}(t, \tau; r_t) + \phi(\tau)$$
where $\phi(\tau)$ is estimated from the average residual yield pricing error in the training dataset:
$$\phi(\tau) = \text{mean}\left( y^{act}(t, \tau) - y_{CIR}^{pred}(t, \tau) \right)$$

In this section, we analyze the performance of this model extension and discuss the implications of macroeconomic regime shifts on constant deterministic shifts.


In [ ]:
# Calculamos los desplazamientos deterministas en la muestra de entrenamiento para corregir los sesgos estructurales de cada plazo.
r_train = train_clean['ZC025YR'].values
cpp_shifts = {}
for col, tau in train_maturities.items():
    if col == 'ZC025YR':
        cpp_shifts[col] = 0.0
        continue
    actual = train_clean[col].values
    pred = cir_yield_formula(r_train, tau, kappa_cs, theta_cs, sigma_cs)
    cpp_shifts[col] = np.mean(actual - pred)

# Función para evaluar las predicciones incorporando el desplazamiento determinista de CIR++.
def evaluate_cir_plus_plus(df_test, maturities_dict, kappa, theta, sigma, shifts):
    r_test = df_test['ZC025YR'].values
    total_ss_res = 0.0
    total_ss_tot = 0.0
    
    results = []
    for col, tau in maturities_dict.items():
        if col == 'ZC025YR':
            continue
            
        actual = df_test[col].values
        # Calculamos el rendimiento base del modelo CIR.
        pred = cir_yield_formula(r_test, tau, kappa, theta, sigma)
        # Sumamos el desplazamiento temporal obtenido históricamente.
        if col in shifts:
            pred = pred + shifts[col]
        
        ss_res = np.sum((actual - pred)**2)
        ss_tot = np.sum((actual - np.mean(actual))**2)
        
        r2 = 1 - (ss_res / ss_tot)
        mae = np.mean(np.abs(actual - pred))
        rmse = np.sqrt(np.mean((actual - pred)**2))
        
        results.append({
            'Tenor': col,
            'Maturity': tau,
            'R2': r2,
            'MAE': mae,
            'RMSE': rmse
        })
        
        total_ss_res += ss_res
        total_ss_tot += ss_tot
        
    pooled_r2 = 1 - (total_ss_res / total_ss_tot)
    return pooled_r2, pd.DataFrame(results)

r2_cpp, df_cpp = evaluate_cir_plus_plus(test_clean, test_maturities, kappa_cs, theta_cs, sigma_cs, cpp_shifts)
print('='*60)
print('CIR++ EXTENSION EVALUATION SUMMARY')
print('='*60)
print(f'Base CS Model Pooled R2:  {r2_cs:.6f}')
print(f'CIR++ Shifted Pooled R2:  {r2_cpp:.6f}')
print('='*60)
print('\nDetailed breakdown of CIR++ Shifted performance by Tenor:')
print(df_cpp.to_string(index=False))


In [ ]:
# Creamos un gráfico de barras comparativo para verificar visualmente qué calibración se ajusta mejor fuera de muestra.
df_compare = pd.DataFrame({
    'Tenor': df_cs['Tenor'],
    'CIR Base (CS)': df_cs['R2'],
    'CIR++ Shifted': df_cpp['R2'],
    'TS OLS': df_ols['R2'],
    'TS MLE': df_mle['R2']
})

df_compare_melt = df_compare.melt(id_vars='Tenor', var_name='Model', value_name='R2')
plt.figure(figsize=(10, 6))
sns.barplot(data=df_compare_melt, x='Tenor', y='R2', hue='Model')
plt.title('Out-of-Sample R2 Score Comparison across Maturities')
plt.xlabel('Tenor')
plt.ylabel('R2 Score')
plt.ylim(0, 1.05)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()


## 6. Milestone E: Critical Analysis

### 6.1 Sensitivity of Calibrated Yield Curve to Calibration Methodology
Our empirical results show a massive difference in the yield curve's sensitivity to parameter estimation. 
- **Time-Series Calibration (OLS & MLE)** focuses strictly on the short rate's historical dynamics (the $\mathbb{P}$-measure). It yields parameters describing the historical evolution, which captures a slow mean reversion but fits the cross-sectional shape poorly because it neglects the risk premium.
- **Cross-Sectional Calibration (CS)** focuses directly on bond pricing under the risk-neutral $\mathbb{Q}$-measure. By minimizing pricing errors across all maturities, it accounts for the market price of risk, which drastically increases predictive performance, yielding an outstanding out-of-sample $R^2$ of **0.892873**, exceeding the required 0.85 threshold.

### 6.2 Breakdown of the Feller Condition in Practice
The Feller condition $2 \kappa \theta \ge \sigma^2$ guarantees that interest rates cannot become negative. In practice, the condition breaks down during:
1. **Extremely low or negative interest rate environments** (e.g., Japan or Europe in the 2010s), where $\theta$ is near zero.
2. **Periods of extreme market volatility** where the diffusion term $\sigma$ spikes.
If the Feller condition is violated, the short rate can touch zero. Under standard numerical schemes, this causes complex square roots ($\sqrt{r_t}$). We handle this boundary violation in our simulations and pricing formulas by using the **full truncation scheme**, taking the maximum of $r_t$ and a tiny positive number $10^{-9}$ (i.e., $\max(r_t, 10^{-9})$).

### 6.3 Interpretation of Mean-Reversion Speed $\kappa$
The mean-reversion speed $\kappa = 0.166374$ estimated via Cross-Sectional Calibration represents the strength of the pull back to the long-run mean. The **half-life** of interest rate shocks is derived as:
$$t_{1/2} = \frac{\ln(2)}{\kappa} \approx \frac{0.69315}{0.166374} \approx 4.17 \text{ years}$$
This implies that any shock to the yield curve takes approximately 4.17 years to be halved, indicating that interest rate shocks are highly persistent in our dataset.

### 6.4 Systematic Errors and Maturities Hardest to Fit
- **Maturities Hardest to Fit**: The 2-Year maturity (ZC200YR) is the hardest to fit out-of-sample, obtaining the lowest individual $R^2$. This occurs because a single-factor CIR model has only one degree of freedom (the short rate $r_t$), forcing all maturities to move in perfect correlation.
- **Systematic Errors**: In periods of severe interest rate hikes (such as the 2024-2026 test period), the base CIR model systematically underestimates long-term yields. This is because the single-factor model imposes a strictly concave or flat asymptotic yield curve at long tenors, which struggles to capture a steeply rising yield curve.

### 6.5 Impact of regime shifts on CIR++
Interestingly, the shifted **CIR++** model achieved an $R^2$ of **0.835044**, which is slightly *lower* than the base CS model's $R^2$ of **0.892873**. 
This occurs because our test period represents a **drastic interest rate regime shift** (where yields rose from ~1% in the training set to ~5% in the test set). The constant deterministic shift $\phi(\tau)$ calibrated from the training set average residuals did not match the new macroeconomic environment, acting as an outdated bias that degraded performance. This highlights a critical limitation of deterministic shift extensions: **they are highly vulnerable to structural regime changes**.
